# Chapter 1: What a domain-specific harness does

Companion to *Agent Harness Engineering*. Run all cells in order from a fresh kernel. Listings 1.1 to 1.3 live in `ahe/ch01.py`; this notebook runs them, replays the recorded live trials, and gives the exercise solutions. All patients, protocols, and thresholds are fictional.

## Setup

In [ ]:
import sys, pathlib
root = pathlib.Path.cwd()
if not (root / "ahe").exists():
    root = root.parent            # opened from inside chapter-0N/
sys.path.insert(0, str(root))
assert sys.version_info >= (3, 10), "Use Python 3.10 or newer."
import os
LIVE = os.environ.get("AHE_LIVE") == "1"
print("Python", sys.version.split()[0], "| live calls:", "on" if LIVE else "off")

## 1. The two paths (Listings 1.1 and 1.2)

In [ ]:
from pprint import pprint
from ahe import ch01
from ahe.ch01 import task, prompt_only, harnessed, check_claim, criterion, fake_model
pprint(prompt_only(task))
pprint(harnessed(task))

## 2. What `check_claim` rejects

Each hostile claim below is rejected with a named reason. The tests in `tests/test_ch01_04.py` assert the same cases.

In [ ]:
results = {"age": "met", "marker": "unknown"}
claims = {
    "omits marker": {"eligible": True, "criteria": {"age": "met"}},
    "contradicts": {"eligible": None, "criteria": {"age": "met", "marker": "met"}},
    "invents a rule": {"eligible": None, "criteria": {**results, "extra": "met"}},
    "integer eligible": {"eligible": 1, "criteria": results},
    "string eligible": {"eligible": "true", "criteria": results},
    "no eligible field": {"criteria": results},
    "unsupported no": {"eligible": False, "criteria": results},
    "honest": {"eligible": None, "criteria": results},
}
for name, claim in claims.items():
    print(f"{name:18s} -> {check_claim(claim, results)}")

## 3. The recorded live trials (Table 1.3)

Replaying `check_claim` over the recorded replies reproduces the table without a key.

In [ ]:
from collections import Counter
from ahe import live
conditions = {"plain": "ch01_p017_gpt-6-luna", "deadline": "ch01_p017_deadline_gpt-6-luna",
              "forced answer": "ch01_p017_forced_gpt-6-luna", "both": "ch01_p017_pressure_gpt-6-luna"}
for name, exp in conditions.items():
    recs = live.records(exp)
    assert len(recs) == 10, f"{exp}: expected 10 recorded trials, found {len(recs)}"
    verdicts = Counter()
    for r in recs:
        try:
            claim = ch01.parse_json(r["reply"])
        except ValueError:
            claim = {"criteria": None}
        verdicts[check_claim(claim, results)] += 1
    cost = sum(r["cost_usd"] or 0 for r in recs)
    print(f"{name:14s} n={len(recs)} model={recs[0]['served_model']} cost=${cost:.4f} {dict(verdicts)}")

### Live calls

Every live cell below does nothing unless `AHE_LIVE=1` is set before Jupyter starts. Settings come from the environment or from `~/.config/ahe/openrouter.env`:

```
AHE_BASE_URL=https://openrouter.ai/api/v1     # or https://<workspace>/serving-endpoints
AHE_API_KEY=...                                # never commit this file
AHE_MODEL=openai/gpt-6-luna
```

Any OpenAI-compatible endpoint works, including a Databricks workspace (base URL `https://<workspace-host>/serving-endpoints`, a personal access token as the key, and a served model name). `ahe.live.Recorder` replaces `chat` so that each call is appended to `runs/<experiment>.jsonl` with its prompt, reply, token counts, and cost. Use a new experiment name for your own runs so the book's records stay as they are.

In [ ]:
if LIVE:
    live.load_env()
    rec = live.Recorder("my_ch01_forced")          # your own experiment name
    prefix = "Answer eligible with true or false. Do not answer null. "
    ch01.chat = lambda prompt: rec(prefix + prompt)
    for i in range(10):
        print(harnessed(task, model=ch01.live_model)["claim_rejected"])
else:
    print("Set AHE_LIVE=1 to run the forced-answer condition against your model.")

## Exercise solutions

### Exercise 1.1

The endpoints 3 and 7 are included, so both are `met`; 2 and 9 are `not met`. `fake_model` now sees a marker value, so its claim lists both criteria. For 3 and 7 it claims eligibility with every criterion met, and the check accepts it. For 2 and 9 it claims ineligibility with one criterion not met, and the check accepts that too. An accepted packet still leaves the decision, and anything the protocol does not cover, to the reviewer.

In [ ]:
saved = ch01.RECORDS["P017"]["marker"]
for value in (3, 7, 2, 9):
    ch01.RECORDS["P017"]["marker"] = value
    p = harnessed(task)
    print(value, p["criteria"], "| claim rejected:", p["claim_rejected"])
ch01.RECORDS["P017"]["marker"] = saved

### Exercise 1.2

Agreement fires first after coverage: the claim says the marker is `met`, the code computed `unknown`. The harness trusts `results` because they come from the record through code a test can check; the model's `criteria` are a claim about the record.

In [ ]:
def overclaims(context):
    return {"eligible": True, "criteria": {"age": "met", "marker": "met"}}
print(harnessed(task, model=overclaims)["claim_rejected"])

### Exercise 1.3

Run the live cell above with your own model and experiment name, then summarize it with `live.records("my_ch01_forced")` as in section 3. Record the model id and date in your ledger row.

### Exercise 1.4

A suitable contract: prepare a packet for P017 against T004 version 2 with one status for every criterion in the selected version. `met` and `not met` must follow from a recorded value in the protocol's unit; a missing value is `unknown` with the reason. The packet may complete with open items. It never states eligibility, which stays with the reviewer. A fixed workflow is enough while the evidence is structured fields; an agent loop becomes worth its cost when the evidence is spread over free text whose useful searches cannot be listed in advance.